# Comparative Genomics

---

## Learning Objectives

By the end of this notebook you will be able to:

- Understand the goals and rationale of comparative genomics
- Implement dot plots from scratch and interpret visual patterns (inversions, duplications, translocations)
- Understand synteny analysis and conserved gene order
- Distinguish orthologs from paralogs and know how to find them
- Navigate genome browsers (UCSC, Ensembl, NCBI)
- Understand whole-genome alignment concepts
- Apply these tools to real biological questions (human vs. mouse, E. coli strain comparison)

## Why this notebook matters

Comparative genomics is how we discover function through conservation. The logic is: if a sequence is conserved across millions of years of evolution, it must be under functional constraint (selection is purifying mutations). This framework identifies regulatory elements (conserved non-coding sequences), dates gene duplications, reveals horizontal gene transfer events, and characterizes the pan-genome of bacterial species. This notebook covers dot plots, synteny analysis, ortholog/paralog detection, genome rearrangements, and whole-genome alignment concepts.

## How to use this notebook

1. Run cells top-to-bottom in order — later cells depend on earlier ones
2. Run the environment check cell first to confirm all imports work
3. Read the explanatory text before each code cell — the context matters
4. The exercises at the end are designed to be done after reading each section

## Complicated moments explained

- **Ortholog vs. paralog in comparative genomics**: Orthologs (genes that diverged via speciation) tend to retain the same function — they are useful for functional inference across species. Paralogs (genes that diverged via duplication) may gain new functions. The most reliable ortholog inference combines sequence similarity with syntenic context — genes that are similar AND in the same genomic neighborhood are more likely to be genuine orthologs.
- **Dot plot diagonals encode structural information**: A continuous diagonal from bottom-left to top-right = conserved linear segment. A diagonal running bottom-right to top-left = conserved but inverted (inversion). Repeated diagonals = tandem repeats. An off-main-diagonal block = transposition. The slope and length of diagonals encode the nature of the genomic relationship.
- **Percent identity thresholds for orthologs are heuristic**: Commonly used thresholds (>30% identity for bacterial orthologs, >70% for strain-level comparison) are empirical rules of thumb. The correct approach is to use synteny, phylogeny, and functional data together. For bacterial species, the species-level boundary is typically defined as <5% ANI (Average Nucleotide Identity) divergence.
- **Core genome vs. pan-genome**: The core genome consists of genes present in all strains of a species. The pan-genome = core + accessory (some strains) + unique (single strains). Bacterial pan-genomes are 'open' (they keep growing as more strains are sequenced); eukaryotic pan-genomes are 'closed'.
- **MUMmer vs. LASTZ vs. minimap2**: For large-genome alignment, sequence length and divergence guide tool choice. MUMmer/nucmer is fast and good for closely related genomes (>90% ANI). LASTZ handles >70% identity. minimap2 is the modern choice for any pairwise comparison, especially with long-read data.

In [ ]:
## Environment check (run this first)

# Environment check
import numpy as np
import matplotlib.pyplot as plt
import shutil

print("Imports ready.")

# Check for comparative genomics tools
tools = ['mummer', 'nucmer', 'lastz', 'minimap2', 'mafft']
for tool in tools:
    found = shutil.which(tool)
    status = f"found at {found}" if found else "NOT FOUND"
    print(f"  {tool}: {status}")
    
print("\nNote: Most demonstrations in this notebook use only Python (numpy/matplotlib)")
print("and do not require external tools.")

In [ ]:
def filtered_dotplot(seq1, seq2, window=5, stringency=0.6):
    """
    Dot plot with sliding window filter.

    Parameters
    ----------
    seq1, seq2 : str
        Sequences to compare.
    window : int
        Window size.
    stringency : float
        Fraction of positions in the window that must match (0.0 to 1.0).

    Returns
    -------
    2D numpy array with match scores.
    """
    threshold = int(window * stringency)
    matrix = np.zeros((len(seq2), len(seq1)))

    for i in range(len(seq2) - window + 1):
        for j in range(len(seq1) - window + 1):
            matches = sum(1 for k in range(window) if seq1[j + k] == seq2[i + k])
            if matches >= threshold:
                ci = i + window // 2
                cj = j + window // 2
                matrix[ci, cj] = matches / window

    return matrix


# Compare different window sizes
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, w in zip(axes, [1, 3, 5]):
    if w == 1:
        mat = simple_dotplot(seq1, seq2).astype(float)
    else:
        mat = filtered_dotplot(seq1, seq2, window=w, stringency=0.6)
    ax.imshow(mat, cmap='Blues', aspect='auto', interpolation='nearest')
    ax.set_title(f'Window = {w}')
    ax.set_xlabel('Sequence 1')
    ax.set_ylabel('Sequence 2')

plt.suptitle('Effect of Window Size on Dot Plot Noise', y=1.02)
plt.tight_layout()
plt.show()

print("Larger windows remove noise but may also lose short regions of similarity.")

### 2.3 Detecting Genomic Rearrangements

Dot plots are powerful for visualizing structural changes. Let us construct synthetic sequences with known rearrangements and see how they appear.

In [ ]:
np.random.seed(42)

def random_dna(length):
    """Generate a random DNA sequence."""
    return ''.join(np.random.choice(list('ACGT'), size=length))


def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    comp = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(comp.get(c, c) for c in reversed(seq))


# Build a reference genome of ~600 bp with distinct segments
seg_A = random_dna(150)  # Segment A
seg_B = random_dna(150)  # Segment B
seg_C = random_dna(150)  # Segment C
seg_D = random_dna(150)  # Segment D

reference = seg_A + seg_B + seg_C + seg_D  # A-B-C-D

# Rearranged genome: A - C_inverted - B - D  (inversion of B-C region)
rearranged = seg_A + reverse_complement(seg_C) + seg_B + seg_D

print(f"Reference genome:  A -- B -- C -- D  ({len(reference)} bp)")
print(f"Rearranged genome: A -- C'-- B -- D  ({len(rearranged)} bp)")
print(f"  (C' = reverse complement of C)")
print(f"\nThis represents an inversion of segment C and a translocation of B and C.")

In [ ]:
def dna_dotplot(seq1, seq2, window=11, stringency=0.7, check_revcomp=True):
    """
    Create a dot plot for DNA sequences, optionally detecting both
    forward and reverse-complement matches.

    Returns (forward_matrix, revcomp_matrix) -- both are 2D arrays.
    """
    threshold = int(window * stringency)
    fwd = np.zeros((len(seq2), len(seq1)))
    rev = np.zeros((len(seq2), len(seq1)))

    seq2_rc = reverse_complement(seq2) if check_revcomp else ''

    for i in range(len(seq2) - window + 1):
        for j in range(len(seq1) - window + 1):
            # Forward match
            fwd_matches = sum(1 for k in range(window) if seq1[j+k] == seq2[i+k])
            if fwd_matches >= threshold:
                fwd[i + window//2, j + window//2] = fwd_matches / window

            # Reverse complement match
            if check_revcomp:
                rev_i = len(seq2) - 1 - i
                rev_matches = sum(1 for k in range(window) if seq1[j+k] == seq2_rc[i+k])
                if rev_matches >= threshold:
                    rev[rev_i - window//2, j + window//2] = rev_matches / window

    return fwd, rev


fwd_mat, rev_mat = dna_dotplot(reference, rearranged, window=11, stringency=0.7)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Forward matches
axes[0].imshow(fwd_mat, cmap='Blues', aspect='auto', interpolation='nearest')
axes[0].set_title('Forward matches')
axes[0].set_xlabel('Reference (A-B-C-D)')
axes[0].set_ylabel('Rearranged (A-C\'-B-D)')

# Reverse complement matches
axes[1].imshow(rev_mat, cmap='Reds', aspect='auto', interpolation='nearest')
axes[1].set_title('Reverse complement matches')
axes[1].set_xlabel('Reference (A-B-C-D)')
axes[1].set_ylabel('Rearranged (A-C\'-B-D)')

# Combined
combined = np.zeros((*fwd_mat.shape, 3))
combined[:, :, 2] = np.clip(fwd_mat, 0, 1)      # Blue = forward
combined[:, :, 0] = np.clip(rev_mat, 0, 1)       # Red = reverse complement
axes[2].imshow(combined, aspect='auto', interpolation='nearest')
axes[2].set_title('Combined (blue=forward, red=inverted)')
axes[2].set_xlabel('Reference (A-B-C-D)')
axes[2].set_ylabel('Rearranged (A-C\'-B-D)')

# Mark segment boundaries
for ax in axes:
    for pos in [150, 300, 450]:
        ax.axvline(pos, color='grey', linewidth=0.5, linestyle='--')
        ax.axhline(pos, color='grey', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Segment A maps to A (diagonal in top-left)")
print("  - Segment C' (inverted C) shows as an anti-diagonal (red)")
print("  - Segment B has shifted position (off-diagonal forward match)")
print("  - Segment D maps to D (diagonal in bottom-right)")

### 2.4 Self-Comparison Dot Plots

Comparing a sequence against itself reveals **internal repeats** and **palindromic sequences** (important for DNA regulatory elements).

In [ ]:
def self_dotplot(sequence, window=7, stringency=0.8):
    """
    Self-comparison dot plot with the main diagonal removed.
    Off-diagonal signals reveal internal repeats.
    """
    matrix = filtered_dotplot(sequence, sequence, window, stringency)

    # Remove trivial main diagonal
    for i in range(min(matrix.shape)):
        lo = max(0, i - window)
        hi = min(matrix.shape[1], i + window + 1)
        matrix[i, lo:hi] = 0

    return matrix


# Create a protein with two repeated domains
domain = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDAT"
linker = "GGSGGSGGSGGS"
protein_with_repeats = domain + linker + domain + linker + domain

matrix = self_dotplot(protein_with_repeats, window=5, stringency=0.7)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(matrix, cmap='Reds', aspect='equal', interpolation='nearest')
ax.set_xlabel('Sequence position')
ax.set_ylabel('Sequence position')
ax.set_title('Self-comparison Dot Plot\n(off-diagonal = internal repeats)')

# Mark domain boundaries
for pos in [len(domain), len(domain) + len(linker),
            2 * len(domain) + len(linker), 2 * len(domain) + 2 * len(linker)]:
    ax.axvline(pos, color='grey', linewidth=0.5, linestyle='--')
    ax.axhline(pos, color='grey', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.show()

print(f"Protein length: {len(protein_with_repeats)} aa")
print(f"Domain length: {len(domain)} aa")
print(f"The parallel off-diagonal lines show 3 copies of the repeated domain.")

---

## 3. Synteny Analysis

**Synteny** refers to the conservation of gene order and orientation between chromosomes of different species. Micro-synteny looks at a few neighboring genes; macro-synteny considers entire chromosomes.

```
Human chr 17:       --[TP53]--[WRAP53]--[EFNB3]--[DLGAP1]--[ATP1B2]--
                        |        |         |         |          |
Mouse chr 11:       --[Trp53]--[Wrap53]--[Efnb3]--[Dlgap1]--[Atp1b2]--

=> The gene order is conserved: this region is syntenic.


Human chr 2:        --[HOXD13]--[HOXD12]--[HOXD11]--[HOXD10]--
                        |          |          |          |
Mouse chr 2:        --[Hoxd13]--[Hoxd12]--[Hoxd11]--[Hoxd10]--

=> Perfect synteny in the HOX cluster -- strong evolutionary constraint.
```

Synteny breaks indicate **rearrangements** (inversions, translocations, fusions, fissions) that occurred after species diverged.

In [ ]:
# Synteny analysis with simulated gene maps

# Species A: chromosome with genes in order
species_a_genes = [
    {'name': 'GeneA', 'start': 100, 'end': 500, 'strand': '+'},
    {'name': 'GeneB', 'start': 800, 'end': 1200, 'strand': '+'},
    {'name': 'GeneC', 'start': 1500, 'end': 2000, 'strand': '+'},
    {'name': 'GeneD', 'start': 2200, 'end': 2700, 'strand': '-'},
    {'name': 'GeneE', 'start': 3000, 'end': 3500, 'strand': '+'},
    {'name': 'GeneF', 'start': 3800, 'end': 4200, 'strand': '+'},
    {'name': 'GeneG', 'start': 4500, 'end': 5000, 'strand': '-'},
    {'name': 'GeneH', 'start': 5300, 'end': 5800, 'strand': '+'},
]

# Species B: same genes but with an inversion of D-E-F region
species_b_genes = [
    {'name': 'GeneA', 'start': 200, 'end': 600, 'strand': '+'},
    {'name': 'GeneB', 'start': 900, 'end': 1300, 'strand': '+'},
    {'name': 'GeneC', 'start': 1600, 'end': 2100, 'strand': '+'},
    # Inverted region: F-E-D (reversed order, flipped strands)
    {'name': 'GeneF', 'start': 2300, 'end': 2700, 'strand': '-'},
    {'name': 'GeneE', 'start': 3000, 'end': 3500, 'strand': '-'},
    {'name': 'GeneD', 'start': 3800, 'end': 4300, 'strand': '+'},
    # Back to normal
    {'name': 'GeneG', 'start': 4600, 'end': 5100, 'strand': '-'},
    {'name': 'GeneH', 'start': 5400, 'end': 5900, 'strand': '+'},
]


def find_ortholog_pairs(genes_a, genes_b):
    """Match genes by name (simulating ortholog identification)."""
    name_to_b = {g['name']: g for g in genes_b}
    pairs = []
    for ga in genes_a:
        gb = name_to_b.get(ga['name'])
        if gb:
            pairs.append((ga, gb))
    return pairs


def detect_synteny_blocks(pairs):
    """
    Detect synteny blocks: consecutive runs of genes in the same
    relative order and orientation.
    """
    blocks = []
    current_block = [pairs[0]]

    for i in range(1, len(pairs)):
        prev_a, prev_b = pairs[i - 1]
        curr_a, curr_b = pairs[i]

        # Check if order is maintained and strand relationship is consistent
        same_order = (curr_b['start'] > prev_b['start'])
        same_strand_rel = ((curr_a['strand'] == curr_b['strand']) ==
                           (prev_a['strand'] == prev_b['strand']))

        if same_order and same_strand_rel:
            current_block.append(pairs[i])
        else:
            blocks.append(current_block)
            current_block = [pairs[i]]

    blocks.append(current_block)
    return blocks


pairs = find_ortholog_pairs(species_a_genes, species_b_genes)
blocks = detect_synteny_blocks(pairs)

print("Synteny Analysis")
print("=" * 65)
print(f"Ortholog pairs found: {len(pairs)}")
print(f"Synteny blocks: {len(blocks)}")
print()

for i, block in enumerate(blocks, 1):
    gene_names = [ga['name'] for ga, _ in block]
    gene_names_b = [gb['name'] for _, gb in block]
    # Check if block is inverted
    inverted = (block[0][0]['strand'] != block[0][1]['strand'])
    status = 'INVERTED' if inverted else 'COLLINEAR'
    print(f"Block {i} ({status}):")
    print(f"  Species A: {' -> '.join(gene_names)}")
    print(f"  Species B: {' -> '.join(gene_names_b)}")

In [ ]:
def plot_synteny(genes_a, genes_b, pairs, title='Synteny Plot'):
    """Visualize synteny between two sets of genes."""
    fig, ax = plt.subplots(figsize=(12, 5))

    y_a = 1.0  # y-position for species A
    y_b = 0.0  # y-position for species B
    gene_height = 0.08

    # Normalization
    max_a = max(g['end'] for g in genes_a)
    max_b = max(g['end'] for g in genes_b)
    scale = max(max_a, max_b)

    colors = plt.cm.Set3(np.linspace(0, 1, len(genes_a)))
    gene_color = {g['name']: colors[i] for i, g in enumerate(genes_a)}

    # Draw chromosomes as lines
    ax.plot([0, max_a / scale], [y_a, y_a], 'k-', linewidth=2)
    ax.plot([0, max_b / scale], [y_b, y_b], 'k-', linewidth=2)

    # Draw genes as arrows
    for g in genes_a:
        x = g['start'] / scale
        w = (g['end'] - g['start']) / scale
        color = gene_color.get(g['name'], 'grey')
        ax.add_patch(plt.Rectangle((x, y_a - gene_height / 2), w, gene_height,
                                   facecolor=color, edgecolor='black', linewidth=0.5))
        ax.text(x + w / 2, y_a + 0.06, g['name'], ha='center', va='bottom', fontsize=7)

    for g in genes_b:
        x = g['start'] / scale
        w = (g['end'] - g['start']) / scale
        color = gene_color.get(g['name'], 'grey')
        ax.add_patch(plt.Rectangle((x, y_b - gene_height / 2), w, gene_height,
                                   facecolor=color, edgecolor='black', linewidth=0.5))
        ax.text(x + w / 2, y_b - 0.08, g['name'], ha='center', va='top', fontsize=7)

    # Draw lines connecting orthologs
    for ga, gb in pairs:
        mid_a = (ga['start'] + ga['end']) / 2 / scale
        mid_b = (gb['start'] + gb['end']) / 2 / scale
        color = gene_color.get(ga['name'], 'grey')
        # Check if inverted
        inverted = ga['strand'] != gb['strand']
        linestyle = '--' if inverted else '-'
        ax.plot([mid_a, mid_b], [y_a - gene_height / 2, y_b + gene_height / 2],
                color=color, linewidth=1.5, linestyle=linestyle, alpha=0.7)

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.3, 1.3)
    ax.set_yticks([y_b, y_a])
    ax.set_yticklabels(['Species B', 'Species A'])
    ax.set_xlabel('Genomic position (normalized)')
    ax.set_title(title)
    ax.text(1.01, y_a, 'Solid = collinear', transform=ax.get_yaxis_transform(),
            fontsize=7, va='center')
    ax.text(1.01, y_b, 'Dashed = inverted', transform=ax.get_yaxis_transform(),
            fontsize=7, va='center')

    plt.tight_layout()
    plt.show()


plot_synteny(species_a_genes, species_b_genes, pairs,
             title='Synteny: Species A vs. Species B (inversion of D-E-F)')

---

## 4. Orthology and Paralogy

Understanding the evolutionary relationship between genes is fundamental to comparative genomics.

```
+--------------------------------------------------------------------------+
|                     ORTHOLOGS vs. PARALOGS                                |
+--------------------------------------------------------------------------+
|                                                                          |
|  Ancestral gene                                                          |
|       |                                                                  |
|       +-------+--------+                                                 |
|       |  duplication   |                                                 |
|       v                v                                                 |
|    Gene_alpha       Gene_beta     <-- PARALOGS (same species, after dup) |
|       |                |                                                 |
|    speciation       speciation                                           |
|    /      \         /      \                                             |
|   v        v       v        v                                            |
| Human_a  Mouse_a Human_b  Mouse_b                                       |
|                                                                          |
|  Human_a <-> Mouse_a : ORTHOLOGS (same gene, different species)          |
|  Human_b <-> Mouse_b : ORTHOLOGS                                        |
|  Human_a <-> Human_b : PARALOGS  (duplication within lineage)            |
|  Human_a <-> Mouse_b : OUT-PARALOGS (duplication before speciation,      |
|                        then speciation on both copies)                   |
|                                                                          |
+--------------------------------------------------------------------------+
```

**Why does it matter?**
- Orthologs usually retain the same function across species (useful for annotation transfer).
- Paralogs often diverge in function (sub- or neo-functionalization).

### Key Databases

| Database | URL | Description |
|----------|-----|------------|
| OrthoDB | orthodb.org | Hierarchical catalog of orthologs |
| OMA | omabrowser.org | Orthologous MAtrix, pairwise orthologs |
| Ensembl Compara | ensembl.org | Gene trees, homology calls |
| NCBI HomoloGene | ncbi.nlm.nih.gov | Automated ortholog sets |
| InParanoid | inparanoid.sbc.su.se | Pairwise ortholog detection |

In [ ]:
# Simulating ortholog/paralog detection with sequence similarity

# Gene family: globins
GLOBIN_FAMILY = {
    'HBA1_HUMAN':  {'name': 'Hemoglobin alpha 1',   'species': 'human',
                    'sequence': 'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH'},
    'HBA1_MOUSE':  {'name': 'Hemoglobin alpha 1',   'species': 'mouse',
                    'sequence': 'MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH'},
    'HBB_HUMAN':   {'name': 'Hemoglobin beta',      'species': 'human',
                    'sequence': 'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST'},
    'HBB_MOUSE':   {'name': 'Hemoglobin beta',      'species': 'mouse',
                    'sequence': 'MVHLTDAEKAAVNGLWGKVNPDDVGGEALGRLLVVYPWTQRYFDSFGDLSS'},
    'MB_HUMAN':    {'name': 'Myoglobin',            'species': 'human',
                    'sequence': 'MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLK'},
    'MB_MOUSE':    {'name': 'Myoglobin',            'species': 'mouse',
                    'sequence': 'MGLSDGEWQLVLNAWGKVEADLAGHGQEVLIGLFKTHPETLDKFDKFKNLK'},
}


def sequence_identity(seq1, seq2):
    """Calculate percent identity between two aligned sequences of equal length."""
    min_len = min(len(seq1), len(seq2))
    matches = sum(1 for i in range(min_len) if seq1[i] == seq2[i])
    return matches / min_len * 100


def classify_homology(gene1_id, gene2_id, gene_db):
    """
    Classify the relationship between two genes.
    """
    g1 = gene_db[gene1_id]
    g2 = gene_db[gene2_id]
    identity = sequence_identity(g1['sequence'], g2['sequence'])

    same_species = g1['species'] == g2['species']
    same_gene = g1['name'] == g2['name']

    if same_gene and not same_species:
        relationship = 'ORTHOLOG'
    elif not same_gene and same_species:
        relationship = 'PARALOG'
    elif not same_gene and not same_species:
        relationship = 'OUT-PARALOG'
    else:
        relationship = 'SAME GENE'

    return relationship, identity


# Compare all pairs
gene_ids = list(GLOBIN_FAMILY.keys())

print("Globin gene family: homology classification")
print("=" * 75)
print(f"{'Gene 1':<15s} {'Gene 2':<15s} {'Relationship':<15s} {'Identity':>8s}")
print("-" * 75)

from itertools import combinations

for g1_id, g2_id in combinations(gene_ids, 2):
    rel, ident = classify_homology(g1_id, g2_id, GLOBIN_FAMILY)
    print(f"{g1_id:<15s} {g2_id:<15s} {rel:<15s} {ident:>7.1f}%")

---

## 5. Genome Browsers

Genome browsers provide an interactive, graphical interface to explore genome annotations in their chromosomal context.

### 5.1 Major Genome Browsers

| Browser | URL | Strengths |
|---------|-----|----------|
| **UCSC Genome Browser** | genome.ucsc.edu | Huge track selection, custom tracks, comparative genomics (conservation scores, chain/net alignments) |
| **Ensembl** | ensembl.org | Gene models, orthology/paralogy, regulation, variation data |
| **NCBI Genome Data Viewer** | ncbi.nlm.nih.gov/gdv/ | Integration with NCBI databases (RefSeq, dbSNP, ClinVar) |
| **IGV (desktop)** | igv.org | Local data visualization (BAM, VCF, BED files) |

### 5.2 Key Comparative Genomics Tracks (UCSC)

- **Conservation (PhyloP, PhastCons)**: base-level conservation scores from multi-species alignments.
- **Chain/Net**: pairwise whole-genome alignments (e.g., human vs. mouse).
- **Synteny (Net tracks)**: shows which regions in another genome correspond to the viewed region.
- **Multiz alignments**: multiple alignment of many vertebrate genomes.

In [ ]:
# Programmatic access to genome annotations via the UCSC API

import json


def ucsc_api_request(endpoint):
    """
    Query the UCSC Genome Browser REST API.
    API docs: https://api.genome.ucsc.edu/
    """
    url = f'https://api.genome.ucsc.edu/{endpoint}'
    try:
        import urllib.request
        with urllib.request.urlopen(url, timeout=15) as resp:
            return json.loads(resp.read().decode('utf-8'))
    except Exception as e:
        print(f"UCSC API error: {e}")
        return None


# Example: list available genomes
print("Querying UCSC API for available genomes...")
data = ucsc_api_request('list/ucscGenomes')
if data and 'ucscGenomes' in data:
    genomes = data['ucscGenomes']
    # Show a few key genomes
    for key in ['hg38', 'mm39', 'dm6', 'ce11', 'sacCer3']:
        if key in genomes:
            info = genomes[key]
            print(f"  {key}: {info.get('description', 'N/A')} "
                  f"({info.get('organism', 'N/A')})")
else:
    print("  (API unavailable in this environment)")
    print("  Common genome assemblies:")
    print("    hg38  - Human (GRCh38)")
    print("    mm39  - Mouse (GRCm39)")
    print("    dm6   - Drosophila melanogaster")
    print("    ce11  - C. elegans")
    print("    sacCer3 - S. cerevisiae")

---

## 6. Whole-Genome Alignment

Whole-genome alignment extends pairwise sequence alignment to entire genomes. This is computationally challenging because genomes are large and contain rearrangements.

### 6.1 Approaches

```
+--------------------------------------------------------------------------+
|                  WHOLE-GENOME ALIGNMENT STRATEGIES                        |
+--------------------------------------------------------------------------+
|                                                                          |
|  1. ANCHOR-BASED (MUMmer, Minimap2)                                      |
|     a) Find exact matches (MUMs = Maximal Unique Matches)                |
|     b) Chain anchors into collinear blocks                               |
|     c) Extend and refine alignments between anchors                      |
|                                                                          |
|  2. SEED-AND-EXTEND (LASTZ, BLAST-based)                                 |
|     a) Find short seed matches (e.g., 12-mers)                           |
|     b) Extend seeds into local alignments (HSPs)                         |
|     c) Chain and filter                                                  |
|                                                                          |
|  3. MULTIPLE GENOME ALIGNMENT (Cactus, Progressive Mauve)                |
|     a) Build a guide tree from pairwise distances                        |
|     b) Progressively align genomes following the tree                    |
|     c) Handle rearrangements with graph-based approaches                 |
|                                                                          |
+--------------------------------------------------------------------------+

Key tools:
  MUMmer (nucmer/promer)  - Fast, anchor-based, good for closely related genomes
  LASTZ                   - Sensitive, used for UCSC chain/net alignments
  Minimap2                - Very fast, long-read and genome alignment
  Cactus                  - Reference-free multiple genome aligner
  Progressive Mauve       - GUI tool for visualizing genome rearrangements
```

In [ ]:
def find_maximal_unique_matches(seq1, seq2, min_length=10):
    """
    Find Maximal Unique Matches (MUMs) between two sequences.

    A MUM is a substring that:
    - occurs exactly once in seq1 and exactly once in seq2
    - is at least min_length characters long
    - cannot be extended in either direction

    This is a simplified O(n^2) implementation for demonstration.
    Real tools (MUMmer) use suffix trees/arrays for O(n) performance.
    """
    mums = []

    for i in range(len(seq1)):
        for j in range(len(seq2)):
            # Extend match
            k = 0
            while (i + k < len(seq1) and j + k < len(seq2)
                   and seq1[i + k] == seq2[j + k]):
                k += 1

            if k >= min_length:
                match_str = seq1[i:i + k]
                # Check uniqueness (simplified)
                if seq1.count(match_str) == 1 and seq2.count(match_str) == 1:
                    mums.append({
                        'seq1_start': i,
                        'seq2_start': j,
                        'length': k,
                        'sequence': match_str[:20] + ('...' if k > 20 else ''),
                    })

    # Remove overlapping MUMs, keep longest
    mums.sort(key=lambda m: m['length'], reverse=True)
    filtered = []
    used1 = set()
    used2 = set()
    for m in mums:
        pos1 = set(range(m['seq1_start'], m['seq1_start'] + m['length']))
        pos2 = set(range(m['seq2_start'], m['seq2_start'] + m['length']))
        if not (pos1 & used1) and not (pos2 & used2):
            filtered.append(m)
            used1 |= pos1
            used2 |= pos2

    filtered.sort(key=lambda m: m['seq1_start'])
    return filtered


# Example: compare two short "genomes" (heavily simplified)
np.random.seed(99)
genome1 = random_dna(200)
# genome2 shares some regions with genome1, with mutations
genome2 = genome1[:50] + random_dna(30) + genome1[50:130] + genome1[130:200]
# Introduce a few mutations
g2_list = list(genome2)
for idx in [60, 100, 150]:
    if idx < len(g2_list):
        g2_list[idx] = {'A': 'T', 'T': 'C', 'C': 'G', 'G': 'A'}[g2_list[idx]]
genome2 = ''.join(g2_list)

mums = find_maximal_unique_matches(genome1, genome2, min_length=8)

print(f"Genome 1: {len(genome1)} bp")
print(f"Genome 2: {len(genome2)} bp")
print(f"\nMaximal Unique Matches (MUMs), min_length=8:")
print(f"{'Seq1 start':>10}  {'Seq2 start':>10}  {'Length':>6}  Sequence")
print("-" * 55)
for m in mums:
    print(f"{m['seq1_start']:>10}  {m['seq2_start']:>10}  {m['length']:>6}  {m['sequence']}")

---

## 7. Comparative Genomics Databases and Tools

| Resource | URL | Description |
|----------|-----|------------|
| **Ensembl Compara** | ensembl.org | Gene trees, synteny, whole-genome alignments |
| **UCSC Genome Browser** | genome.ucsc.edu | Conservation tracks, chain/net alignments |
| **OrthoDB** | orthodb.org | Hierarchical ortholog catalog |
| **OMA Browser** | omabrowser.org | Ortholog matrix, HOGs |
| **NCBI Genome** | ncbi.nlm.nih.gov/genome | Assembly data, comparative tools |
| **Genomicus** | genomicus.bio.ens.psl.eu | Synteny and ancestral genome browser |
| **SynMap (CoGe)** | genomevolution.org | Online synteny dot plots |
| **D-GENIES** | dgenies.toulouse.inra.fr | Fast dot plot for large genomes |
| **MUMmer** | mummer4.github.io | Whole-genome alignment |
| **Progressive Mauve** | darlinglab.org/mauve | Multiple genome alignment + visualization |

---

## 8. Case Studies

### 8.1 Human vs. Mouse: Conserved Synteny

Human and mouse diverged approximately 90 million years ago. Despite this, large regions of conserved synteny exist:

```
Key human-mouse synteny relationships:

  Human chr 1p  <-> Mouse chr 4
  Human chr 1q  <-> Mouse chr 1
  Human chr 2   <-> Mouse chr 1, 2, 6   (human chr 2 = fusion of two ancestral chr)
  Human chr 6   <-> Mouse chr 10, 17
  Human chr 17  <-> Mouse chr 11
  Human chr 20  <-> Mouse chr 2
  Human chr X   <-> Mouse chr X          (strong conservation)

  ~40% of the human genome can be aligned 1:1 to the mouse genome.
  ~340 synteny blocks between human and mouse (at megabase scale).
```

This conserved synteny is why mouse is such a powerful model organism for human genetics.

In [ ]:
# Simplified human-mouse synteny data (selected chromosomal regions)

human_mouse_synteny = [
    {'human_chr': '1',  'human_start': 0,   'human_end': 120,  'mouse_chr': '4',  'orientation': '+'},
    {'human_chr': '1',  'human_start': 120, 'human_end': 248,  'mouse_chr': '1',  'orientation': '+'},
    {'human_chr': '2',  'human_start': 0,   'human_end': 90,   'mouse_chr': '1',  'orientation': '+'},
    {'human_chr': '2',  'human_start': 90,  'human_end': 140,  'mouse_chr': '2',  'orientation': '-'},
    {'human_chr': '2',  'human_start': 140, 'human_end': 243,  'mouse_chr': '6',  'orientation': '+'},
    {'human_chr': '6',  'human_start': 0,   'human_end': 80,   'mouse_chr': '10', 'orientation': '+'},
    {'human_chr': '6',  'human_start': 80,  'human_end': 170,  'mouse_chr': '17', 'orientation': '-'},
    {'human_chr': '17', 'human_start': 0,   'human_end': 83,   'mouse_chr': '11', 'orientation': '+'},
    {'human_chr': '20', 'human_start': 0,   'human_end': 64,   'mouse_chr': '2',  'orientation': '+'},
    {'human_chr': 'X',  'human_start': 0,   'human_end': 155,  'mouse_chr': 'X',  'orientation': '+'},
]


def plot_oxford_grid(synteny_data, species1='Human', species2='Mouse'):
    """
    Create an Oxford grid (chromosome-level synteny dot plot).
    Each dot represents a synteny block.
    """
    fig, ax = plt.subplots(figsize=(8, 8))

    # Get unique chromosomes
    human_chrs = sorted(set(s['human_chr'] for s in synteny_data),
                        key=lambda c: (0, int(c)) if c.isdigit() else (1, c))
    mouse_chrs = sorted(set(s['mouse_chr'] for s in synteny_data),
                        key=lambda c: (0, int(c)) if c.isdigit() else (1, c))

    h_idx = {c: i for i, c in enumerate(human_chrs)}
    m_idx = {c: i for i, c in enumerate(mouse_chrs)}

    colors = plt.cm.tab10(np.linspace(0, 1, len(human_chrs)))

    for s in synteny_data:
        x = h_idx[s['human_chr']]
        y = m_idx[s['mouse_chr']]
        size = (s['human_end'] - s['human_start']) * 3  # scale
        color = colors[h_idx[s['human_chr']]]
        marker = 'o' if s['orientation'] == '+' else 'D'
        ax.scatter(x, y, s=size, c=[color], marker=marker, edgecolors='black',
                   linewidths=0.5, alpha=0.8)

    ax.set_xticks(range(len(human_chrs)))
    ax.set_xticklabels([f'chr{c}' for c in human_chrs], rotation=45)
    ax.set_yticks(range(len(mouse_chrs)))
    ax.set_yticklabels([f'chr{c}' for c in mouse_chrs])
    ax.set_xlabel(f'{species1} chromosomes')
    ax.set_ylabel(f'{species2} chromosomes')
    ax.set_title(f'{species1} vs. {species2} Synteny (Oxford Grid)')
    ax.grid(True, alpha=0.3)

    # Legend
    ax.scatter([], [], marker='o', c='grey', label='Same orientation (+)', edgecolors='black')
    ax.scatter([], [], marker='D', c='grey', label='Inverted (-)', edgecolors='black')
    ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()


plot_oxford_grid(human_mouse_synteny)

print("Each dot = a synteny block.")
print("Dots on the same row = regions of one mouse chromosome found on different human chromosomes.")
print("Human chr 2 maps to mouse chr 1, 2, and 6 (known ancestral fusion event).")

### 8.2 E. coli Strain Comparison

Comparing different strains of the same bacterial species reveals:
- **Core genome**: genes present in all strains
- **Accessory genome**: genes present in some strains (often on genomic islands)
- **Unique genes**: strain-specific genes (phage insertions, plasmids)

This **pan-genome** concept is central to bacterial comparative genomics.

In [ ]:
# Simulated E. coli pan-genome analysis

# Gene content of different E. coli strains
ecoli_strains = {
    'K-12 MG1655': {
        'description': 'Laboratory strain, non-pathogenic',
        'genome_size_mb': 4.64,
        'total_genes': 4290,
        'genes': set(range(1, 4291))   # gene IDs 1-4290 (core genes)
    },
    'O157:H7 EDL933': {
        'description': 'Enterohemorrhagic (EHEC), causes bloody diarrhea',
        'genome_size_mb': 5.53,
        'total_genes': 5324,
        'genes': set(range(1, 4291)) | set(range(5001, 6035))  # core + pathogenicity genes
    },
    'CFT073': {
        'description': 'Uropathogenic (UPEC), causes urinary tract infections',
        'genome_size_mb': 5.23,
        'total_genes': 5379,
        'genes': set(range(1, 4291)) | set(range(6001, 7090))  # core + UTI virulence genes
    },
    'Nissle 1917': {
        'description': 'Probiotic strain',
        'genome_size_mb': 5.10,
        'total_genes': 5043,
        'genes': set(range(1, 4291)) | set(range(7001, 7754))  # core + fitness genes
    },
}

# Remove some core genes from each strain (simulate gene loss)
np.random.seed(42)
for strain_name, strain in ecoli_strains.items():
    lost = set(np.random.choice(list(range(3800, 4291)), size=50, replace=False))
    strain['genes'] -= lost


def pan_genome_analysis(strains):
    """Compute core, accessory, and unique gene sets."""
    all_genes = set()
    for s in strains.values():
        all_genes |= s['genes']

    # Core = present in ALL strains
    core = all_genes.copy()
    for s in strains.values():
        core &= s['genes']

    # Unique = present in exactly ONE strain
    unique_per_strain = {}
    for name, s in strains.items():
        others = set()
        for other_name, other_s in strains.items():
            if other_name != name:
                others |= other_s['genes']
        unique_per_strain[name] = s['genes'] - others

    # Accessory = not core and not unique
    accessory = all_genes - core
    for u in unique_per_strain.values():
        accessory -= u

    return {
        'pan_genome_size': len(all_genes),
        'core_size': len(core),
        'accessory_size': len(accessory),
        'unique': {name: len(genes) for name, genes in unique_per_strain.items()},
    }


result = pan_genome_analysis(ecoli_strains)

print("E. coli Pan-Genome Analysis (4 strains)")
print("=" * 55)
print(f"Pan-genome size:  {result['pan_genome_size']:,} genes")
print(f"Core genome:      {result['core_size']:,} genes "
      f"({result['core_size']/result['pan_genome_size']*100:.1f}%)")
print(f"Accessory genome: {result['accessory_size']:,} genes")
print(f"\nStrain-specific (unique) genes:")
for name, count in result['unique'].items():
    desc = ecoli_strains[name]['description']
    print(f"  {name:<20s}  {count:>5} unique genes  ({desc})")

In [ ]:
def plot_pan_genome(strains, analysis_result):
    """Visualize pan-genome composition."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: bar chart of gene counts per strain
    names = list(strains.keys())
    total_genes = [len(strains[n]['genes']) for n in names]
    core_count = analysis_result['core_size']

    x = np.arange(len(names))
    axes[0].bar(x, total_genes, color='steelblue', label='Total genes')
    axes[0].axhline(core_count, color='red', linestyle='--', label=f'Core genome ({core_count})')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=8)
    axes[0].set_ylabel('Number of genes')
    axes[0].set_title('Gene Count per Strain')
    axes[0].legend()

    # Right: pie chart of pan-genome composition
    total_unique = sum(analysis_result['unique'].values())
    sizes = [analysis_result['core_size'], analysis_result['accessory_size'], total_unique]
    labels_pie = [f'Core\n({sizes[0]})', f'Accessory\n({sizes[1]})', f'Unique\n({sizes[2]})']
    colors_pie = ['#2ecc71', '#f39c12', '#e74c3c']

    axes[1].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
                startangle=90, textprops={'fontsize': 9})
    axes[1].set_title('Pan-Genome Composition')

    plt.tight_layout()
    plt.show()


plot_pan_genome(ecoli_strains, result)

---

## 9. Exercises

### Exercise 1: Build a Dot Plot from Scratch

Create a filtered dot plot comparing the two DNA sequences below. Try window sizes of 5, 9, and 15. Which window size gives the clearest signal?

In [ ]:
# Exercise 1

dna_seq1 = "ATGCGATCGATCGATCGATCGATCGATCGATCGAATTCCGGAATTCCGGAATTCCGGATCGATCGATCGATCGATCG"
dna_seq2 = "ATGCGATCGATCGATCGATCGATCGATCGATCGTTAACCGGTTAACCGGTTAACCGGATCGATCGATCGATCGATCG"

# YOUR CODE HERE:
# 1. Create filtered dot plots with window=5, 9, and 15
# 2. Display them side by side
# 3. Describe what you see


In [ ]:
# Solution 1

dna_seq1 = "ATGCGATCGATCGATCGATCGATCGATCGATCGAATTCCGGAATTCCGGAATTCCGGATCGATCGATCGATCGATCG"
dna_seq2 = "ATGCGATCGATCGATCGATCGATCGATCGATCGTTAACCGGTTAACCGGTTAACCGGATCGATCGATCGATCGATCG"

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, w in zip(axes, [5, 9, 15]):
    mat = filtered_dotplot(dna_seq1, dna_seq2, window=w, stringency=0.7)
    ax.imshow(mat, cmap='Blues', aspect='auto', interpolation='nearest')
    ax.set_title(f'Window = {w}')
    ax.set_xlabel('Sequence 1')
    ax.set_ylabel('Sequence 2')

plt.suptitle('Dot Plot: Effect of Window Size', y=1.02)
plt.tight_layout()
plt.show()

print("Observations:")
print("  - The two sequences share identical flanking regions (ATCGATCG repeats)")
print("  - The middle region differs (AATTCCGG vs TTAACCGG)")
print("  - Window=5 shows the repeat structure clearly but with some noise")
print("  - Window=9 is a good compromise between sensitivity and specificity")
print("  - Window=15 is very clean but may miss shorter features")

### Exercise 2: Identify Genomic Rearrangements

The code below creates two synthetic genomes. One has a **duplication** of segment B. Create a dot plot and identify the duplication visually.

In [ ]:
# Exercise 2

np.random.seed(123)
seg_X = random_dna(100)
seg_Y = random_dna(100)
seg_Z = random_dna(100)

genome_original = seg_X + seg_Y + seg_Z       # X-Y-Z
genome_duplicated = seg_X + seg_Y + seg_Y + seg_Z  # X-Y-Y-Z (Y is duplicated)

# YOUR CODE HERE:
# 1. Create a dot plot of genome_original (x-axis) vs genome_duplicated (y-axis)
# 2. Identify the duplication pattern in the plot
# 3. Also create a self-dot-plot of genome_duplicated to see the internal repeat


In [ ]:
# Solution 2

np.random.seed(123)
seg_X = random_dna(100)
seg_Y = random_dna(100)
seg_Z = random_dna(100)

genome_original = seg_X + seg_Y + seg_Z
genome_duplicated = seg_X + seg_Y + seg_Y + seg_Z

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pairwise comparison
mat1 = filtered_dotplot(genome_original, genome_duplicated, window=9, stringency=0.7)
axes[0].imshow(mat1, cmap='Blues', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Original (X-Y-Z)')
axes[0].set_ylabel('Duplicated (X-Y-Y-Z)')
axes[0].set_title('Pairwise: Original vs Duplicated')
# Mark segment boundaries
for pos in [100, 200]:
    axes[0].axvline(pos, color='red', linewidth=0.5, linestyle='--')
for pos in [100, 200, 300]:
    axes[0].axhline(pos, color='red', linewidth=0.5, linestyle='--')

# Self-comparison of duplicated genome
mat2 = self_dotplot(genome_duplicated, window=9, stringency=0.7)
axes[1].imshow(mat2, cmap='Reds', aspect='equal', interpolation='nearest')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Position')
axes[1].set_title('Self-comparison: Duplicated (X-Y-Y-Z)')
for pos in [100, 200, 300]:
    axes[1].axvline(pos, color='grey', linewidth=0.5, linestyle='--')
    axes[1].axhline(pos, color='grey', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.show()

print("Pairwise plot:")
print("  The Y region in the original maps to TWO parallel diagonals in the duplicated genome.")
print("  This is the hallmark of a tandem duplication.")
print("\nSelf-comparison:")
print("  An off-diagonal line connecting positions 100-200 and 200-300 shows")
print("  that the duplicated genome has an internal repeat (the two copies of Y).")

### Exercise 3: Find Orthologs Between Two Species

Given the gene content of two fictitious species, identify ortholog pairs based on sequence identity (>60% identity = likely ortholog).

In [ ]:
# Exercise 3

species_x_genes = {
    'X_kinase1':  'MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDAT',
    'X_kinase2':  'MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDAT',  # paralog of kinase1
    'X_receptor': 'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWT',
    'X_enzyme':   'MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPE',
}

species_y_genes = {
    'Y_kinase':   'MSKGEELFAGVVPILVELdgdvnghkfsvsgesegdat',  # ortholog of X_kinase1/2
    'Y_receptor': 'MVHLTAEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWT',  # ortholog of X_receptor
    'Y_enzyme':   'MGLSDGEWQLVLNAWGKVEADLAGHGQEVLIGLFKTHPE',  # ortholog of X_enzyme
    'Y_novel':    'QSKTEVNDHRQQMPLLWRFIAGAVVTLLGATLIQQFPTP',  # no ortholog in species X
}

# YOUR CODE HERE:
# 1. Compare every gene in species X against every gene in species Y
# 2. Calculate sequence identity using the sequence_identity() function
# 3. Identify ortholog pairs (identity > 60%)
# 4. Identify genes unique to each species


In [ ]:
# Solution 3

print("All pairwise sequence identities:")
print(f"{'Gene X':<15s} {'Gene Y':<15s} {'Identity':>8s} {'Ortholog?':>10s}")
print("-" * 50)

ortholog_pairs = []
for gx_name, gx_seq in species_x_genes.items():
    for gy_name, gy_seq in species_y_genes.items():
        ident = sequence_identity(gx_seq, gy_seq)
        is_orth = ident > 60
        if is_orth:
            ortholog_pairs.append((gx_name, gy_name, ident))
        if ident > 30:  # only print non-trivial matches
            print(f"{gx_name:<15s} {gy_name:<15s} {ident:>7.1f}% {'YES' if is_orth else '':>10s}")

print(f"\nIdentified {len(ortholog_pairs)} ortholog pairs.")

# Genes without orthologs
x_with_orthologs = {p[0] for p in ortholog_pairs}
y_with_orthologs = {p[1] for p in ortholog_pairs}

x_unique = set(species_x_genes.keys()) - x_with_orthologs
y_unique = set(species_y_genes.keys()) - y_with_orthologs

print(f"\nGenes unique to Species X (no ortholog): {x_unique or 'none'}")
print(f"Genes unique to Species Y (no ortholog): {y_unique or 'none'}")

print("\nNote: X_kinase1 and X_kinase2 are paralogs that both match Y_kinase.")
print("This is a 'one-to-many' orthology (or co-orthologs), common after lineage-specific duplication.")

### Exercise 4: Synteny Block Detection

Species C has the gene order A-B-C-D-E-F-G-H. Species D has A-B-F-E-D-C-G-H (an inversion of the C-D-E-F block). Write code to detect the inversion.

In [ ]:
# Exercise 4

species_c_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
species_d_order = ['A', 'B', 'F', 'E', 'D', 'C', 'G', 'H']

# YOUR CODE HERE:
# 1. Compare the gene orders
# 2. Identify which genes are in the same order (collinear blocks)
# 3. Identify which genes are in reversed order (inverted blocks)
# 4. Report the breakpoints of the inversion


In [ ]:
# Solution 4

species_c_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
species_d_order = ['A', 'B', 'F', 'E', 'D', 'C', 'G', 'H']

# Map gene to position in species C
c_pos = {gene: i for i, gene in enumerate(species_c_order)}

# Get positions in C for each gene in D's order
d_in_c_coords = [c_pos[gene] for gene in species_d_order]

print("Gene order analysis:")
print(f"Species C: {' - '.join(species_c_order)}")
print(f"Species D: {' - '.join(species_d_order)}")
print(f"\nD genes mapped to C positions: {d_in_c_coords}")

# Find blocks of consecutive increasing or decreasing positions
blocks = []
current = [0]
for i in range(1, len(d_in_c_coords)):
    diff = d_in_c_coords[i] - d_in_c_coords[i - 1]
    if len(current) == 1:
        current.append(i)
        continue
    prev_diff = d_in_c_coords[current[-1]] - d_in_c_coords[current[-2]]
    if (diff > 0 and prev_diff > 0) or (diff < 0 and prev_diff < 0):
        current.append(i)
    else:
        blocks.append(current)
        current = [current[-1], i]
blocks.append(current)

print("\nDetected blocks:")
for block in blocks:
    genes = [species_d_order[i] for i in block]
    positions = [d_in_c_coords[i] for i in block]
    direction = 'COLLINEAR' if positions[-1] > positions[0] else 'INVERTED'
    print(f"  {' - '.join(genes):20s}  C positions: {positions}  -> {direction}")

print("\nConclusion: genes C-D-E-F are inverted in species D (appear as F-E-D-C).")
print("Inversion breakpoints: between B and C, and between F and G in species C.")

### Exercise 5: Pan-Genome Venn Diagram

Using the `ecoli_strains` data, compute the number of genes shared by every combination of 2 strains. Create a matrix showing pairwise shared gene counts.

In [ ]:
# Exercise 5

# YOUR CODE HERE:
# 1. For each pair of strains, compute the number of shared genes
# 2. Display as a matrix (heatmap or table)
# 3. Which pair of strains shares the most genes? The fewest?


In [ ]:
# Solution 5

strain_names = list(ecoli_strains.keys())
n = len(strain_names)

# Compute pairwise shared gene counts
shared_matrix = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(n):
        shared = ecoli_strains[strain_names[i]]['genes'] & ecoli_strains[strain_names[j]]['genes']
        shared_matrix[i, j] = len(shared)

# Print as table
print("Pairwise shared gene counts:")
header = f"{'':>20s}" + ''.join(f"{n[:12]:>14s}" for n in strain_names)
print(header)
print("-" * len(header))
for i, name in enumerate(strain_names):
    row = f"{name[:20]:>20s}" + ''.join(f"{shared_matrix[i, j]:>14d}" for j in range(n))
    print(row)

# Heatmap
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(shared_matrix, cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(n))
ax.set_xticklabels([s[:12] for s in strain_names], rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n))
ax.set_yticklabels([s[:12] for s in strain_names], fontsize=8)
ax.set_title('Pairwise Shared Genes Between E. coli Strains')
plt.colorbar(im, label='Shared genes')

# Annotate cells
for i in range(n):
    for j in range(n):
        ax.text(j, i, str(shared_matrix[i, j]), ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

# Find min/max off-diagonal
min_val = float('inf')
max_val = 0
min_pair = max_pair = ('', '')
for i in range(n):
    for j in range(i + 1, n):
        if shared_matrix[i, j] < min_val:
            min_val = shared_matrix[i, j]
            min_pair = (strain_names[i], strain_names[j])
        if shared_matrix[i, j] > max_val:
            max_val = shared_matrix[i, j]
            max_pair = (strain_names[i], strain_names[j])

print(f"\nMost shared genes:  {max_pair[0]} & {max_pair[1]} ({max_val} genes)")
print(f"Fewest shared genes: {min_pair[0]} & {min_pair[1]} ({min_val} genes)")

### Exercise 6: Conservation Score Plot

Simulate a PhyloP-like conservation score along a genomic region. Given the aligned sequences of 5 species, compute per-position conservation (fraction of species with the same base as the reference) and plot it.

In [ ]:
# Exercise 6

# Simulated multiple alignment of 200 bp from 5 species
np.random.seed(55)
ref_seq = random_dna(200)

# Other species have mutations at random positions
def mutate(seq, mutation_rate):
    """Randomly mutate a sequence."""
    bases = list('ACGT')
    result = list(seq)
    for i in range(len(result)):
        if np.random.random() < mutation_rate:
            result[i] = np.random.choice([b for b in bases if b != result[i]])
    return ''.join(result)

aligned_species = {
    'Human':   ref_seq,
    'Chimp':   mutate(ref_seq, 0.02),  # very close
    'Mouse':   mutate(ref_seq, 0.15),  # moderate divergence
    'Chicken': mutate(ref_seq, 0.30),  # distant
    'Zebrafish': mutate(ref_seq, 0.40),  # very distant
}

# YOUR CODE HERE:
# 1. For each position, compute the fraction of species matching the human base
# 2. Plot the conservation score along the sequence
# 3. Identify the most and least conserved regions


In [ ]:
# Solution 6

np.random.seed(55)
ref_seq = random_dna(200)

def mutate(seq, mutation_rate):
    bases = list('ACGT')
    result = list(seq)
    for i in range(len(result)):
        if np.random.random() < mutation_rate:
            result[i] = np.random.choice([b for b in bases if b != result[i]])
    return ''.join(result)

aligned_species = {
    'Human':   ref_seq,
    'Chimp':   mutate(ref_seq, 0.02),
    'Mouse':   mutate(ref_seq, 0.15),
    'Chicken': mutate(ref_seq, 0.30),
    'Zebrafish': mutate(ref_seq, 0.40),
}

# Compute per-position conservation
species_names = list(aligned_species.keys())
seqs = [aligned_species[s] for s in species_names]
ref = seqs[0]
n_species = len(seqs)

conservation = []
for pos in range(len(ref)):
    matches = sum(1 for s in seqs if s[pos] == ref[pos])
    conservation.append(matches / n_species)

conservation = np.array(conservation)

# Smooth with a sliding window
window = 10
smoothed = np.convolve(conservation, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].bar(range(len(conservation)), conservation, width=1.0, color='steelblue', alpha=0.7)
axes[0].set_ylabel('Conservation score')
axes[0].set_title('Per-position conservation (fraction matching human)')
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5)

axes[1].plot(range(window // 2, window // 2 + len(smoothed)), smoothed,
             color='darkgreen', linewidth=2)
axes[1].fill_between(range(window // 2, window // 2 + len(smoothed)), smoothed,
                     alpha=0.3, color='green')
axes[1].set_ylabel(f'Smoothed (window={window})')
axes[1].set_xlabel('Position')

plt.tight_layout()
plt.show()

# Statistics
print(f"Mean conservation: {conservation.mean():.2f}")
print(f"Fully conserved positions (score=1.0): {(conservation == 1.0).sum()} / {len(conservation)}")
print(f"Highly conserved regions (smoothed > 0.8):")
in_region = False
for i, val in enumerate(smoothed):
    pos = i + window // 2
    if val > 0.8 and not in_region:
        start = pos
        in_region = True
    elif val <= 0.8 and in_region:
        print(f"  {start}-{pos} ({pos - start} bp)")
        in_region = False
if in_region:
    print(f"  {start}-{len(ref)} ({len(ref) - start} bp)")

---

## Summary

| Topic | Key Concept |
|-------|------------|
| Comparative genomics | Comparing genomes to understand evolution and function |
| Dot plots | Visual sequence comparison; diagonals=homology, anti-diagonals=inversion |
| Window filtering | Reduce noise by requiring multiple matches in a sliding window |
| Synteny | Conserved gene order between species |
| Orthologs | Same gene in different species (speciation event) |
| Paralogs | Related genes within a species (duplication event) |
| Genome browsers | UCSC, Ensembl, NCBI -- interactive genome visualization |
| Whole-genome alignment | MUMmer (anchor-based), LASTZ (seed-extend), Cactus (multi-genome) |
| Pan-genome | Core + accessory + unique genes across strains |

## Resources

- [UCSC Genome Browser](https://genome.ucsc.edu/) -- conservation, chain/net tracks
- [Ensembl Compara](https://www.ensembl.org/info/genome/compara/index.html) -- gene trees, synteny
- [OrthoDB](https://www.orthodb.org/) -- ortholog database
- [OMA Browser](https://omabrowser.org/) -- orthology prediction
- [MUMmer](https://mummer4.github.io/) -- whole-genome alignment
- [Minimap2](https://github.com/lh3/minimap2) -- fast genome alignment
- [D-GENIES](http://dgenies.toulouse.inra.fr/) -- web-based dot plot for large genomes
- [Progressive Mauve](http://darlinglab.org/mauve/) -- multiple genome alignment
- [Genomicus](https://www.genomicus.bio.ens.psl.eu/) -- synteny browser

---[< Previous: Gene Ontology and Pathway Analysis](../11_Gene_Ontology_and_Pathways/02_pathways.ipynb) | [Tier 2: Core Bioinformatics Overview](../README.md) | [Next: Computational Genetics >](../13_Computational_Genetics/01_computational_genetics.ipynb)---